In [ ]:
# STEP 1 - Convert pdf into separate pages
from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader

#loader = WebBaseLoader("https://en.wikipedia.org/wiki/Climate_change")
#loader = TextLoader("AI.txt")

loader = PyPDFLoader("Artificial-Intelligence.pdf")

pdf_pages = loader.load()
print(f'No of pages: {len(pdf_pages)}')

In [ ]:
# STEP 2 - Convert every page into sub-pages (chunks)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ",", " "]
)
chunk_documents = splitter.split_documents(pdf_pages)
print(f'No of chunks: {len(chunk_documents)}')

In [ ]:
# STEP 3 - Convert every chunk into vectors.
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = OllamaEmbeddings(model="nomic-embed-text")
print('Embedding model is ready')

faiss_vector_store = FAISS.from_documents(chunk_documents, embedding_model)
faiss_vector_store.save_local("faiss-ai-book")
print('FAISS Vector store created')

In [ ]:
# STEP 4 - Build Retrieval Chain

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

model = ChatOllama(model="gpt-oss:120b-cloud")
print('Chat Model is ready to use!')

USER_PROMPT_TEMPLATE = """Use the following piece of the context to answer user's question.
If you don't know the answer, just say you don't know, don't try to make the answer.
---------------------------------------
{context}
Question: {question}
"""

prompt_template = PromptTemplate(
    template=USER_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)
qa_chain = RetrievalQA.from_chain_type(
    model,
    chain_type="stuff",
    retriever=faiss_vector_store.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt_template}
)
print('Retrieval chain is ready!!')

In [ ]:
response = qa_chain.invoke("What is an Artificial Intelligence?")
response

In [ ]:
response = qa_chain.invoke("What are the key points about algorithms?")
response

In [ ]:
# STEP 5: Build UI interface

import gradio as gr

def chatbot_response(message, history):
    # Use the created RetrievalQA chain to get the answer
    response = qa_chain({"query": message})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Format the response to include the answer and source documents (optional)
    formatted_response = f"{answer}" # You can add source documents here if desired

    print(f"Source documents: {source_documents}")

    return formatted_response

# Create the Gradio interface
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="RAG AI Chatbot",
    description="Ask questions about AI."
)

# Launch the interface
iface.launch(share=True, server_port=7861)